# Document Grounding SDK – Vector, Retrieval и Pipelines

This notebook demonstrates the changes introduced to the SDK:

- **Vector API**: management of collections/documents and semantic search.
- **Retrieval API**: search across configured repositories (vector / help.sap.com, etc.).
- **Pipelines API**: added missing methods (search, executions, documents, trigger, etc.).

> The examples below are written so they can be executed in environment with access configured to SAP GenAI Hub / AI Core.


In [ ]:
from __future__ import annotations

import json
from typing import Any

def as_json(obj: Any) -> str:
    """Helper pretty-print for Pydantic models"""
    if hasattr(obj, 'model_dump'):
        data = obj.model_dump(by_alias=True, exclude_none=True)
    else:
        data = obj
    return json.dumps(data, ensure_ascii=False, indent=2)


## Client Initialization

Separate clients were added under `gen_ai_hub.document_grounding.clients.*`, while backward compatibility is preserved via re-exports in `gen_ai_hub.document_grounding.client`.

### 1) List and search pipelines

In [ ]:
from gen_ai_hub.proxy import get_proxy_client

# Backwards compatible imports
from gen_ai_hub.document_grounding.client import PipelineAPIClient, VectorAPIClient, RetrievalAPIClient

proxy_client = get_proxy_client(proxy_version='gen-ai-hub')

pipelines = PipelineAPIClient(proxy_client)
vector = VectorAPIClient(proxy_client)
retrieval = RetrievalAPIClient(proxy_client)

print('Clients initialized')


## Pipelines API

Below are examples of the main operations. There were introduced the following methods:

- `search_pipelines(...)`
- `get_pipeline_executions(...)`, `get_pipeline_execution_by_id(...)`
- `get_execution_documents(...)`, `get_execution_document_by_id(...)`
- `get_pipeline_documents(...)`, `get_pipeline_document_by_id(...)`
- `trigger_pipeline(...)`




In [ ]:
pipelines_list = pipelines.get_pipelines(top=10)
print(as_json(pipelines_list))

from gen_ai_hub.document_grounding.models.pipeline import SearchPipelineRequest, SearchPipelineData

search_req = SearchPipelineRequest(data=SearchPipelineData(search='<SEARCH_QUERY>'))
search_res = pipelines.search_pipelines(search_req)
print(as_json(search_res))

### 2) Pipeline status and manual trigger

To start a pipeline, use `trigger_pipeline` (Manual Trigger)

In [ ]:
from gen_ai_hub.document_grounding.models.pipeline import ManualPipelineTrigger

pipeline_id = '<PIPELINE_ID>'

status = pipelines.get_pipeline_status(pipeline_id)
print('Status:')
print(as_json(status))

trigger_req = ManualPipelineTrigger()
trigger_res = pipelines.trigger_pipeline(pipeline_id, trigger_req)
print('Trigger response:')
print(as_json(trigger_res))


### 3) Executions and Documents

There were added coverage for working with pipeline executions and documents.
This is useful for diagnostics: which documents were processed, what errors occurred, etc.


In [ ]:
execs = pipelines.get_pipeline_executions(pipeline_id, top=20)
print(as_json(execs))

execution_id = '<EXECUTION_ID>'
execution = pipelines.get_pipeline_execution_by_id(pipeline_id, execution_id)
print(as_json(execution))


docs = pipelines.get_execution_documents(pipeline_id, execution_id, top=50)
print(as_json(docs))


document_id = '<DOCUMENT_ID>'
doc = pipelines.get_execution_document_by_id(pipeline_id, execution_id, document_id)
print(as_json(doc))


pipeline_docs = pipelines.get_pipeline_documents(pipeline_id, top=50)
print(as_json(pipeline_docs))


pipeline_doc = pipelines.get_pipeline_document_by_id(pipeline_id, document_id)
print(as_json(pipeline_doc))


## Vector API

The Vector API is designed for managing **collections** and **documents**, as well as performing search.

### 1) List collections

In [ ]:
collections = vector.get_collections(top=50)
print(as_json(collections))


### 2) Create a collection

In `CollectionCreateRequest`, the `embeddingConfig` field is required.


In [ ]:
from gen_ai_hub.document_grounding.models.vector import CollectionCreateRequest, EmbeddingConfig

create_req = CollectionCreateRequest(
    title='My SDK Demo Collection',
    embeddingConfig=EmbeddingConfig(modelName='text-embedding-3-large'),
    metadata=[],
)

create_res = vector.create_collection(create_req)
print(as_json(create_res))


### 3) Add / update / delete documents

In [ ]:
from gen_ai_hub.document_grounding.models.vector import (
    DocumentsCreateRequest, DocumentsUpdateRequest,
    TextOnlyBaseChunk, BaseDocument, VectorKeyValueListPair
)

collection_id = '<COLLECTION_ID>'

doc = BaseDocument(
    chunks=[TextOnlyBaseChunk(content='Hello from SDK Vector API', metadata=[])],
    metadata=[VectorKeyValueListPair(key='source', value=['notebook'])],
)

create_docs_req = DocumentsCreateRequest(documents=[doc])
created = vector.create_documents(collection_id, create_docs_req)
print('Created:')
print(as_json(created))

documents = vector.get_documents(collection_id, top=20)
print(as_json(documents))
document_id = documents.resources[0].id

update_req = DocumentsUpdateRequest(documents=[ ... ])
updated = vector.update_documents(collection_id, update_req)
print(as_json(updated))

vector.delete_document(collection_id, document_id)


### 4) Search across collections (Text Search)

`TextSearchRequest` is used with `filters`, where `collectionIds` and limits for chunks/documents are specified.

In [ ]:
from gen_ai_hub.document_grounding.models.vector import (
    TextSearchRequest, VectorSearchFilter, VectorSearchConfiguration
)

search_req = TextSearchRequest(
    query='Hello',
    filters=[
        VectorSearchFilter(
            id='f1',
            collectionIds=[collection_id],
            configuration=VectorSearchConfiguration(maxChunkCount=5, maxDocumentCount=3),
            documentMetadata=[],
            chunkMetadata=[],
            collectionMetadata=[],
        )
    ],
)

search_res = vector.search(search_req)
print(as_json(search_res))


## Retrieval API

The Retrieval API provides a unified search interface across configured data repositories (e.g., `vector` collections or `help.sap.com`).

### 1) List repositories

In [ ]:
repos = retrieval.get_data_repositories(top=50)
print(as_json(repos))

repo_id = '<DATA_REPOSITORY_ID>'
repo = retrieval.get_data_repository_by_id(repo_id)
print(as_json(repo))


### 2) Retrieval search

In `RetrievalSearchFilter`, the `dataRepositoryType` is specified (e.g., `'vector'` or `'help.sap.com'`), along with optional constraints/metadata.

The example below shows a typical request structure.

In [ ]:
from gen_ai_hub.document_grounding.models.retrieval import (
    RetrievalSearchInput, RetrievalSearchFilter, RetrievalSearchConfiguration
)

retrieval_req = RetrievalSearchInput(
    query='How to configure Document Grounding?',
    filters=[
        RetrievalSearchFilter(
            id='r1',
            dataRepositoryType='help.sap.com',
            searchConfiguration=RetrievalSearchConfiguration(maxChunkCount=5, maxDocumentCount=3),
            dataRepositories=[],
        )
    ],
)

retrieval_res = retrieval.search(retrieval_req)
print(as_json(retrieval_res))
